# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

DECISION_MONTH = "2026-03"
START_DATE = "2026-03-01"
END_DATE = "2026-03-31"

FACT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Setup complete.")
print("Decision month:", DECISION_MONTH)

Setup complete.
Decision month: 2026-03


### Feature vector

I use five observed page-level features from the March 2026 decision window. These features are available at decision time because they come from observed GSC and GA4 performance during the decision window. Content and client identifiers are retained only as context and are not used as predictive features.

In [12]:
# Build the March 2026 page-level frame

feature_frame = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0
        END
    ) AS gsc_impressions,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS gsc_clicks,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )
            /
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
        ELSE NULL
    END AS gsc_avg_position,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN COALESCE(ga4_sessions, 0)
            ELSE 0
        END
    ) AS ga4_sessions,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN COALESCE(ga4_engaged_sessions, 0)
            ELSE 0
        END
    ) AS ga4_engaged_sessions

FROM read_parquet('{FACT_PATH}')

WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'

GROUP BY
    content_hash_id,
    client_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.298246,0.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,6.893301,1.0,0.0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,5.637584,4.0,0.0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,3.214128,0.0,0.0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.535346,3.0,0.0


In [13]:
FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

CONTEXT_FIELDS = [
    "content_hash_id",
    "client_hash_id",
]

feature_vector = feature_frame[
    CONTEXT_FIELDS + FEATURES
].copy()

for col in FEATURES:
    feature_vector[col] = pd.to_numeric(
        feature_vector[col],
        errors="coerce"
    )

feature_vector["gsc_avg_position"] = (
    feature_vector["gsc_avg_position"].fillna(
        feature_vector["gsc_avg_position"].median()
    )
)

for col in [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "ga4_engaged_sessions",
]:
    feature_vector[col] = feature_vector[col].fillna(0)

print("Feature vector shape:", feature_vector.shape)

print("\nFeatures:")
for col in FEATURES:
    print("-", col)

print("\nMissing values after filling:")
display(feature_vector[FEATURES].isna().sum())

display(feature_vector.head(10))

Feature vector shape: (331437, 7)

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Missing values after filling:


,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
ga4_engaged_sessions,0


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.298246,0.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,6.893301,1.0,0.0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,5.637584,4.0,0.0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,3.214128,0.0,0.0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.535346,3.0,0.0
5,content_05434271b257bb68,client_73cda7b4e4f265ea,1421.0,6.0,6.906404,9.0,0.0
6,content_22610b0934f8825e,client_73cda7b4e4f265ea,67.0,0.0,12.000000,0.0,0.0
7,content_712c365258cee05c,client_73cda7b4e4f265ea,6048.0,23.0,4.931878,8.0,0.0
8,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,223.0,1.0,10.538117,2.0,0.0
9,content_1f380a642aed423b,client_73cda7b4e4f265ea,96.0,1.0,5.864583,8.0,0.0


### Feature notes

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `gsc_impressions` | Total observed Google Search Console impressions during March 2026. | Missing values are filled with 0 after checking GSC availability. | Available at decision time from observed search data. |
| `gsc_clicks` | Total observed Google Search Console clicks during March 2026. | Missing values are filled with 0 after checking GSC availability. | Available at decision time from observed search data. |
| `gsc_avg_position` | Weighted average Google Search Console position during March 2026. | Missing values are filled with the feature median. | Available at decision time from observed search data. |
| `ga4_sessions` | Total observed GA4 sessions during March 2026. | Missing values are filled with 0 when GA4 data is unavailable. | Available at decision time when GA4 data is available. |
| `ga4_engaged_sessions` | Total observed GA4 engaged sessions during March 2026. | Missing values are filled with 0 when GA4 data is unavailable. | Available at decision time when GA4 data is available. |

The content and client identifiers are context fields only. They are not predictive features.

In [14]:
print("Feature availability and missing values:")

display(
    feature_vector[FEATURES].isna().sum().to_frame("missing_values")
)

print("Rows:", len(feature_vector))
print("Feature count:", len(FEATURES))

Feature availability and missing values:


,missing_values
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
ga4_engaged_sessions,0


Rows: 331437
Feature count: 5


### Leakage hunt

I checked for the main leakage risks: the target or label-derived fields, future-period measurements, identifiers, and fields that describe the observation window rather than the content opportunity.

The honest feature vector contains only the five observed March performance features. The declining proxy is not included in the feature vector. Future-period measurements are also excluded because they would not be available at the decision moment.

In [15]:
# Leakage checks

LEAKAGE_FIELDS = [
    "is_declining_label",
    "trend_direction",
]

FUTURE_FIELDS = [
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "future_position",
]

print("Checking leakage fields...")

for field in LEAKAGE_FIELDS + FUTURE_FIELDS:
    print(f"{field}:",
          "PRESENT" if field in feature_vector.columns else "NOT PRESENT")

assert "is_declining_label" not in feature_vector.columns
assert "trend_direction" not in feature_vector.columns

for field in FUTURE_FIELDS:
    assert field not in feature_vector.columns

print("\nLeakage checks passed.")

Checking leakage fields...
is_declining_label: NOT PRESENT
trend_direction: NOT PRESENT
future_impressions: NOT PRESENT
future_clicks: NOT PRESENT
future_sessions: NOT PRESENT
future_position: NOT PRESENT

Leakage checks passed.


### Excluded fields

- `is_declining_label` — excluded because it is the label/proxy and using it as a feature would directly leak the answer.
- `trend_direction` — excluded because it is not a decision-time feature and is not present as a valid warehouse field for this development slice.
- Future-period performance fields — excluded because they would not be known at the decision moment.
- `report_date` — excluded because it identifies the daily observation rather than describing the content opportunity.
- `month` — excluded because it identifies the development window rather than the content itself.
- `content_hash_id` — retained only as context for page identification and validation, not as a predictive feature.
- `client_hash_id` — retained only as context for grouping and validation, not as a predictive feature.
- `gsc_data_available` — used to determine whether GSC measurements are available, but not used as a predictive feature.
- `ga4_data_available` — used to determine whether GA4 measurements are available, but not used as a predictive feature.

The final honest feature set contains five observed performance features and excludes label-derived and future information.

In [16]:
EXCLUDED_FIELDS = {
    "is_declining_label": "Label/proxy; would directly leak the answer.",
    "trend_direction": "Not a valid decision-time feature in this warehouse slice.",
    "future_period_measurements": "Would not be available at the decision moment.",
    "report_date": "Identifies the observation window rather than the content opportunity.",
    "month": "Identifies the warehouse month rather than the content opportunity.",
    "content_hash_id": "Context identifier; not used predictively.",
    "client_hash_id": "Context identifier; not used predictively.",
    "gsc_data_available": "Availability flag used for data handling, not prediction.",
    "ga4_data_available": "Availability flag used for data handling, not prediction.",
}

print("Excluded fields and reasons:")
for field, reason in EXCLUDED_FIELDS.items():
    print(f"- {field}: {reason}")

Excluded fields and reasons:
- is_declining_label: Label/proxy; would directly leak the answer.
- trend_direction: Not a valid decision-time feature in this warehouse slice.
- future_period_measurements: Would not be available at the decision moment.
- report_date: Identifies the observation window rather than the content opportunity.
- month: Identifies the warehouse month rather than the content opportunity.
- content_hash_id: Context identifier; not used predictively.
- client_hash_id: Context identifier; not used predictively.
- gsc_data_available: Availability flag used for data handling, not prediction.
- ga4_data_available: Availability flag used for data handling, not prediction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [17]:
assert len(FEATURES) == 5
assert "is_declining_label" not in feature_vector.columns
assert "trend_direction" not in feature_vector.columns

assert feature_vector["content_hash_id"].notna().all()
assert feature_vector["client_hash_id"].notna().all()

assert feature_vector[FEATURES].isna().sum().sum() == 0

print("===================================")
print("ML-05 SELF-CHECK")
print("===================================")
print("Decision month:", DECISION_MONTH)
print("Feature vector rows:", len(feature_vector))
print("Honest feature count:", len(FEATURES))
print("Label leakage excluded: YES")
print("Future fields excluded: YES")
print("Context IDs retained separately: YES")
print("Missing values handled: YES")
print("===================================")
print("Self-check passed.")

ML-05 SELF-CHECK
Decision month: 2026-03
Feature vector rows: 331437
Honest feature count: 5
Label leakage excluded: YES
Future fields excluded: YES
Context IDs retained separately: YES
Missing values handled: YES
Self-check passed.
